# 03 — Évaluation du RAG simple (référence)

Ce notebook mesure la qualité du RAG simple sur le jeu de questions **dev**. Ces résultats servent de référence (*baseline*) pour le système multi-agents.

Le jeu **test** n'est pas utilisé ici : il est réservé à la comparaison finale, pour que les réglages ne soient jamais faits sur les questions qui servent à juger le système.

Les réponses ont été générées avec `scripts/evaluate.py --split dev`, exécuté sur GPU dans un notebook Kaggle (`kaggle/evaluation_kaggle.ipynb`).

In [1]:
import json

import pandas as pd

from assistant_rag.config import RESULTS_DIR
from assistant_rag.evaluation.dataset import load_questions
from assistant_rag.evaluation.metrics import score_records, summarize
from assistant_rag.evaluation.runner import load_records

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 2)

run_dir = RESULTS_DIR / "simple-dev"
config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
records = load_records(run_dir / "answers.jsonl")
scores = score_records(records).set_index("id")
answers = pd.DataFrame(records).set_index("id")

## 1. Jeu de questions

60 questions écrites à partir du texte des deux règlements, chacune avec une réponse de référence et les articles qui la justifient :

- **factuelles** : la réponse se trouve dans un seul article ;
- **multi-articles** : la réponse combine deux articles, parfois des deux règlements ;
- **hors corpus** : la réponse n'est pas dans les documents ; le système doit refuser de répondre.

In [2]:
questions = pd.DataFrame([q.__dict__ for q in load_questions()])
pd.crosstab(questions["type"], questions["split"], margins=True, margins_name="total")

split,dev,test,total
type,,,
factuelle,24,12,36
hors_corpus,8,4,12
multi_articles,8,4,12
total,40,20,60


## 2. Configuration évaluée

In [3]:
settings = config["settings"]
pd.Series(
    {
        "LLM": settings["llm_model"],
        "Embeddings": settings["embed_model"],
        "Taille des chunks": settings["chunk_size"],
        "Passages retrouvés (k)": settings["top_k"],
        "Version du prompt": config["prompt_version"],
        "LLM-juge": config["judge"],
        "Version du prompt du juge": config.get("judge_prompt_version", "juge-v1"),
        "Commit": config["git_commit"],
    },
    name="valeur",
).to_frame()

,valeur
LLM,qwen3:4b-instruct
Embeddings,bge-m3
Taille des chunks,512
Passages retrouvés (k),5
Version du prompt,rag-v1
LLM-juge,qwen3:8b
Version du prompt du juge,juge-v1
Commit,2e8f72c


## 3. Résultats

| Métrique | Définition |
| --- | --- |
| Hit@k | part des questions dont au moins un article attendu figure parmi les k passages retrouvés |
| Rappel@k | part des articles attendus retrouvés |
| MRR | moyenne de 1 / rang du premier article attendu retrouvé |
| Précision des citations | part des passages cités qui appartiennent à un article attendu |
| Rappel des citations | part des articles attendus effectivement cités |
| Exactitude | note du LLM-juge : 1 si correct, 0,5 si partiel, 0 si incorrect ou refus |
| Refus à tort | part des questions avec réponse pour lesquelles le système a refusé de répondre |
| Refus correct | part des questions hors corpus pour lesquelles le système a refusé de répondre |

La précision des citations est stricte : citer un article pertinent mais non listé dans les articles attendus compte comme une erreur.

In [4]:
summarize(records)

,Questions,Hit@k,Rappel@k,MRR,Précision des citations,Rappel des citations,Exactitude,Refus à tort,Refus correct,Latence moyenne (s)
Factuelles,24,1.0,1.00,0.94,0.96,1.00,0.85,0.00,NaN,7.32
Multi-articles,8,1.0,0.81,0.94,0.93,0.75,0.62,0.12,NaN,5.95
Hors corpus,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.44
Ensemble,40,1.0,0.95,0.94,0.95,0.94,0.80,0.03,1.0,6.07


## 4. Exactitude selon le LLM-juge

In [5]:
judged = answers[answers["judge"].notna()]
verdicts = judged["judge"].apply(lambda j: j["verdict"]).rename("verdict")
verdicts.value_counts().rename("questions").to_frame()

,questions
verdict,
correct,21
partiel,9
incorrect,1


## 5. Analyse des erreurs

### 5.1 Recherche : articles attendus absents des passages retrouvés

In [6]:
def articles(items):
    return ", ".join(f"{i['document']} {i['article']}" for i in items)


misses = scores[scores["retrieval_recall"] < 1].index
answers.loc[misses].assign(
    attendus=lambda d: d["expected"].apply(lambda e: ", ".join(f"{doc} {art}" for doc, art in e)),
    retrouvés=lambda d: d["retrieved"].apply(articles),
)[["type", "question", "attendus", "retrouvés"]]

,type,question,attendus,retrouvés
id,,,,
multi-06,multi_articles,Une décision automatisée peut-elle reposer sur des catégories particulières de données comme les données de santé ?,"rgpd 22, rgpd 9","rgpd 22, ai_act 10, ai_act 10, ai_act 3, ai_act 10"
multi-07,multi_articles,Quelles amendes l'AI Act prévoit-il pour les opérateurs de systèmes d'IA et pour les fournisseurs de modèles d'IA à ...,"ai_act 99, ai_act 101","ai_act 101, ai_act 55, ai_act 111, ai_act 50, ai_act 101"
multi-11,multi_articles,Comment le RGPD protège-t-il les données biométriques et quels usages de la biométrie l'AI Act interdit-il ?,"rgpd 9, ai_act 5","ai_act 5, ai_act 5, ai_act 5, ai_act 5, ai_act 5"


### 5.2 Réponses jugées partielles ou incorrectes

In [7]:
wrong = verdicts[verdicts != "correct"].index
answers.loc[wrong].assign(
    verdict=lambda d: d["judge"].apply(lambda j: j["verdict"]),
    justification=lambda d: d["judge"].apply(lambda j: j["justification"]),
    recherche_ok=scores.loc[wrong, "hit"].astype(bool),
)[["question", "verdict", "recherche_ok", "justification"]]

,question,verdict,recherche_ok,justification
id,,,,
rgpd-12,Quelle est la durée minimale du mandat des membres d'une autorité de contrôle ?,partiel,True,La réponse évaluée omet l'exception concernant le premier mandat après le 24 mai 2016 et la possibilité d'une durée ...
aiact-04,Quelles obligations de transparence s'appliquent aux systèmes d'IA qui interagissent avec des personnes ou qui génèr...,partiel,True,La réponse évaluée contient des informations correctes mais inclut des détails supplémentaires non présents dans la ...
aiact-06,Quelle amende est prévue en cas de non-respect des pratiques interdites de l'AI Act ?,partiel,True,La réponse évaluée mentionne des montants d'amende différents de ceux de la référence et ne précise pas correctement...
aiact-08,Quelles amendes la Commission peut-elle infliger aux fournisseurs de modèles d'IA à usage général ?,partiel,True,La réponse évaluée inclut correctement le montant des amendes mais ajoute des détails non mentionnés dans la réponse...
aiact-14,Une personne a-t-elle droit à une explication d'une décision prise sur la base d'un système d'IA à haut risque ?,partiel,True,"La réponse évaluée mentionne correctement le droit à des explications, mais elle inclut une condition non mentionnée..."
aiact-16,À partir de quand l'AI Act est-il applicable ?,partiel,True,"La réponse évaluée indique la date d'application générale de l'AI Act, mais omet les exceptions et les dates précise..."
aiact-18,Comment le marquage CE doit-il être apposé sur un système d'IA à haut risque ?,partiel,True,"La réponse évaluée mentionne correctement la visibilité, la lisibilité et l'indélébilité du marquage CE, ainsi que l..."
multi-04,Quelles informations doivent être fournies à la personne concernée selon que les données sont collectées auprès d'el...,partiel,True,La réponse évaluée inclut des éléments corrects mais omet certaines informations essentielles comme les coordonnées ...
multi-06,Une décision automatisée peut-elle reposer sur des catégories particulières de données comme les données de santé ?,incorrect,True,La réponse évaluée affirme que les décisions automatisées peuvent reposer sur des catégories particulières de donnée...


La colonne `recherche_ok` sépare les deux origines possibles d'une erreur : si le bon article a été retrouvé, l'erreur vient de la génération ; sinon, elle vient de la recherche.

### 5.3 Refus

In [8]:
refusals = scores[(scores["false_refusal"] == 1) | (scores["correct_refusal"] == 0)].index
answers.loc[refusals, ["type", "question", "refused", "answer"]]

,type,question,refused,answer
id,,,,
multi-07,multi_articles,Quelles amendes l'AI Act prévoit-il pour les opérateurs de systèmes d'IA et pour les fournisseurs de modèles d'IA à ...,True,"L'AI Act prévoit des amendes pour les fournisseurs de modèles d’IA à usage général, qui ne peuvent excéder 3 % de le..."


### 5.4 Citations hors des articles attendus

In [9]:
imprecise = scores[scores["citation_precision"] < 1].index
answers.loc[imprecise].assign(
    attendus=lambda d: d["expected"].apply(lambda e: ", ".join(f"{doc} {art}" for doc, art in e)),
    cités=lambda d: d["cited"].apply(articles),
)[["question", "attendus", "cités"]]

,question,attendus,cités
id,,,
rgpd-08,Quelles données sont considérées comme des catégories particulières de données à caractère personnel ?,rgpd 9,"rgpd 9, ai_act 3"
aiact-06,Quelle amende est prévue en cas de non-respect des pratiques interdites de l'AI Act ?,ai_act 99,"ai_act 100, ai_act 99"
multi-06,Une décision automatisée peut-elle reposer sur des catégories particulières de données comme les données de santé ?,"rgpd 22, rgpd 9","rgpd 22, ai_act 3"


## 6. Contrôle manuel du LLM-juge

Le LLM-juge peut se tromper. Les 31 réponses qu'il a notées ont été relues une à une et comparées à la référence et au texte des articles ; le verdict de cette relecture est enregistré dans `revue_humaine.csv`.

In [10]:
review = pd.read_csv(run_dir / "revue_humaine.csv").set_index("id")
agree = review["verdict_juge"] == review["verdict_humain"]
print(f"Accord entre le juge et la relecture : {agree.mean():.0%} ({agree.sum()} / {len(review)})")
pd.crosstab(review["verdict_humain"], review["verdict_juge"], rownames=["relecture"], colnames=["juge"])

Accord entre le juge et la relecture : 74% (23 / 31)


juge,correct,incorrect,partiel
relecture,,,
correct,21,1,5
incorrect,0,0,2
partiel,0,0,2


In [11]:
review.loc[~agree].join(answers["question"])[["question", "verdict_juge", "verdict_humain", "commentaire"]]

,question,verdict_juge,verdict_humain,commentaire
id,,,,
rgpd-12,Quelle est la durée minimale du mandat des membres d'une autorité de contrôle ?,partiel,correct,La durée de quatre ans répond à la question ; l'exception transitoire est un détail.
aiact-04,Quelles obligations de transparence s'appliquent aux systèmes d'IA qui interagissent avec des personnes ou qui génèr...,partiel,correct,"Réponse exacte et plus complète que la référence (textes d'intérêt public, hypertrucages) : le juge pénalise des ajo..."
aiact-06,Quelle amende est prévue en cas de non-respect des pratiques interdites de l'AI Act ?,partiel,incorrect,"Montant faux : 1 500 000 EUR (article 100, institutions de l'Union) au lieu de 35 000 000 EUR ou 7 %."
aiact-08,Quelles amendes la Commission peut-elle infliger aux fournisseurs de modèles d'IA à usage général ?,partiel,correct,Montants exacts ; les conditions ajoutées sont exactes.
aiact-14,Une personne a-t-elle droit à une explication d'une décision prise sur la base d'un système d'IA à haut risque ?,partiel,correct,La condition jugée ajoutée figure bien dans l'article 86.
aiact-18,Comment le marquage CE doit-il être apposé sur un système d'IA à haut risque ?,partiel,correct,Exact ; le numéro de l'organisme notifié est une précision exacte de l'article 48.
multi-06,Une décision automatisée peut-elle reposer sur des catégories particulières de données comme les données de santé ?,incorrect,correct,"« Oui, à condition que l'article 9, paragraphe 2, point a) ou g), s'applique » a le même sens que la référence."
multi-11,Comment le RGPD protège-t-il les données biométriques et quels usages de la biométrie l'AI Act interdit-il ?,partiel,incorrect,Affirmation inventée sur le RGPD : l'article 9 n'a pas été retrouvé et la réponse lui attribue une règle fausse.


Le juge est surtout trop sévère : il classe « partiel » des réponses exactes qui donnent plus de détails que la référence. Il est aussi trop indulgent sur une erreur de montant (aiact-06). Son prompt a été précisé en conséquence (version `juge-v2`) pour les évaluations suivantes.

Exactitude recalculée avec les verdicts de la relecture :

In [12]:
from assistant_rag.evaluation.judge import VERDICT_SCORES

answerable = scores[scores["type"] != "hors_corpus"]
human = answerable["correctness"].copy()
human.loc[review.index] = review["verdict_humain"].map(VERDICT_SCORES)
exactness = pd.DataFrame(
    {
        "juge": answerable.groupby("type")["correctness"].mean(),
        "relecture": human.groupby(answerable["type"]).mean(),
    }
)
exactness.loc["ensemble"] = [answerable["correctness"].mean(), human.mean()]
exactness

,juge,relecture
type,,
factuelle,0.85,0.94
multi_articles,0.62,0.69
ensemble,0.80,0.88


## 7. Temps de réponse sur GPU

In [13]:
scores["latency"].describe(percentiles=[0.5, 0.9]).rename("secondes").to_frame()

,secondes
count,40.00
mean,6.07
std,11.11
min,0.82
50%,4.25
90%,7.94
max,73.14


## 8. Conclusions

**Recherche.** Pour les 32 questions qui ont une réponse, au moins un article attendu figure toujours parmi les 5 passages retrouvés (Hit@5 = 100 %), le plus souvent en première position (MRR = 0,94). Toutes les questions factuelles retrouvent leur article. Le rappel tombe à 81 % sur les questions multi-articles : pour 3 questions sur 8, l'un des deux articles attendus manque, car plusieurs chunks d'un même article long occupent les 5 places.

**Citations.** 95 % des passages cités appartiennent à un article attendu et 94 % des articles attendus sont cités. Les citations hors des articles attendus sont rares et souvent pertinentes (définition de l'AI Act citée pour les données sensibles), sauf pour aiact-06.

**Exactitude.** Selon le juge, l'exactitude est de 0,80. Après relecture des 31 réponses notées, elle est de **0,88** : 0,94 sur les questions factuelles et 0,69 sur les questions multi-articles. Le juge n'est d'accord avec la relecture que dans 74 % des cas, surtout parce qu'il pénalise des réponses exactes plus détaillées que la référence. Son prompt a été corrigé pour la suite, et les comparaisons finales seront de nouveau contrôlées à la main.

**Refus.** Les 8 questions hors corpus sont toutes refusées : le système n'invente pas de réponse lorsque les documents ne la contiennent pas. Une seule question avec réponse est refusée à tort (multi-07) : l'article 99 n'ayant pas été retrouvé, la réponse traite une moitié de la question puis conclut par la phrase de refus.

**Erreurs typiques.**

| Erreur | Exemple | Origine |
| --- | --- | --- |
| Article manquant dans une question à deux volets | multi-07, multi-11 | recherche : un seul article occupe les 5 places |
| Affirmation non justifiée par les passages | multi-11 : règle du RGPD inventée | génération, faute de source |
| Confusion entre articles voisins | aiact-06 : amende de l'article 100 au lieu de l'article 99 | génération |
| Réponse incomplète | aiact-16 : exceptions de calendrier omises | génération |
| Notation perdue à l'extraction | 10^25 lu « 1025 » | extraction du PDF |

**Temps de réponse.** Sur GPU T4, la réponse médiane prend 4,3 secondes (contre 1,5 à 4 minutes sur le processeur du poste de développement).

**Conséquences pour le système multi-agents.** Les erreurs se concentrent sur les questions qui combinent plusieurs articles et sur les affirmations non vérifiées. Le système multi-agents cible ces deux points :

- le **chercheur** décompose la question en sous-questions, lance une recherche pour chacune et limite le nombre de passages par article, pour couvrir tous les volets de la question ;
- le **vérificateur** contrôle que chaque phrase est justifiée par le passage qu'elle cite, et renvoie la réponse au rédacteur dans le cas contraire.